# Tutorial: Topic classification Using simple RNN, LSTM, Bi-LSTM, & GRU

In this tutorial, we will learn how to build RNN model in keras and tensorflow for topic classification.

Steps:
 - Load the dataset (topic categorization dataset)
 - Clean the dataset
 - Text vectorization
 - Model Building, training, and evaluating (Simple RNN, LSTM, BiLSTM, & GRU)
 

## Import the Neccesary Libraries

In [55]:
import re
import warnings
import itertools
import numpy as np 
import pandas as pd 
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Load the dataset

In [56]:
from sklearn.datasets import fetch_20newsgroups

categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

twenty_train = fetch_20newsgroups(subset='train',
    categories=categories, shuffle=True, random_state=42)

twenty_test = fetch_20newsgroups(subset='test',
    categories=categories, shuffle=True, random_state=42)


# 2. Cleaning the data before modeling

In [57]:
import re

def text_cleaning(text):
    text = text.lower() # convert to lower case
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' ', text) # remove emails
    text = re.sub(r'subject:.*',' ',text) # remove lines start with Subject
    text = re.sub(r'write to:.*',' ',text) # remove lines start with Write to:
    text = re.sub(r'from:.*',' ',text) # remove lines start with From:
    
    text = re.sub(r'nntp-posting-host:.*',' ',text) # remove lines start with Nntp-Posting-Host:
    text = re.sub(r'lines:.*',' ',text) # remove lines start with Lines:
    text = re.sub(r'organization:.*',' ',text) # remove lines start with Organization:
    text = re.sub(r'email:.*',' ',text) # remove lines start with email:
    text = re.sub(r'tel:.*',' ',text) # remove lines start with tel:
    text = re.sub(r'fax:.*',' ',text) # remove lines start with fax:
    
    text = re.sub("\(.*?\)","",text) # Delete all the data which are present in the brackets.
    
    # Decontractions, replace words like below to full words.
    text = re.sub(r"can\'t","can not",text)
    text = re.sub(r"\'s","is",text)
    text = re.sub(r"i\'ve'","i have",text)
    text = re.sub(r"i\'m","i am",text)
    text = re.sub(r"you\'re","you are",text)
    text = re.sub(r"i\'ll","i will",text)
    
    text = re.sub(r'[^A-Za-z_]'," ",text) # Replace all the words except "A-Za-z_" with space. 
    text = re.sub(r"[0-9]","",text) # Replace all the digits with space i.e delete all the digits. 
    
    return text


for i, text in enumerate(twenty_train.data):
    twenty_train.data[i] = text_cleaning(text)
    
for i, text in enumerate(twenty_test.data):
    twenty_test.data[i] = text_cleaning(text)

X_train, y_train = twenty_train.data, twenty_train.target
X_test, y_test = twenty_test.data, twenty_test.target


# 3. Text vectorization

In [58]:
vocab_size = 5000
maxlen = 300
vectorizer = tf.keras.layers.TextVectorization(max_tokens=vocab_size, output_sequence_length=maxlen)
vectorizer.adapt(X_train)

# 4. Prepare datasets

In [59]:
batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000)
train_ds = train_ds.batch(batch_size).map(lambda x, y: (vectorizer(x), y)).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(batch_size).map(lambda x, y: (vectorizer(x), y)).prefetch(tf.data.AUTOTUNE)

# 5. Build the simple RNN model


In [60]:
simple_rnn_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, mask_zero=True),
    tf.keras.layers.SpatialDropout1D(0.5),
    tf.keras.layers.SimpleRNN(64),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(categories), activation='softmax')
])

simple_rnn_model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

simple_rnn_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_9                  │ ?                           │               0 │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_7 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_22 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_23 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 6. Train and Evaluate the simple RNN model

In [36]:
simple_rnn_model.fit(train_ds, validation_data=test_ds, epochs=20)

loss, acc = simple_rnn_model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.2734 - loss: 1.3889 - val_accuracy: 0.2843 - val_loss: 1.3848
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.3292 - loss: 1.3356 - val_accuracy: 0.2936 - val_loss: 1.3658
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.4502 - loss: 1.2418 - val_accuracy: 0.3489 - val_loss: 1.3406
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.5525 - loss: 1.0916 - val_accuracy: 0.3948 - val_loss: 1.3104
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.6779 - loss: 0.8870 - val_accuracy: 0.4214 - val_loss: 1.2910
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.7953 - loss: 0.5894 - val_accuracy: 0.5233 - val_loss: 1.1309
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.8737 - loss: 0.4015 - val_accuracy: 0.4874 - val_loss: 1.2688
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.9123 - loss: 0.2844 - val_accuracy: 0.5519 - v

# 7. Build the LSTM model

In [37]:
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, mask_zero=True),
    tf.keras.layers.SpatialDropout1D(0.5),
    tf.keras.layers.LSTM(64, dropout = 0.3, recurrent_dropout = 0.3 ),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(categories), activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_4                  │ ?                           │               0 │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_8 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_9 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 8. Train and Evaluate the LSTM Model

In [38]:
lstm_model.fit(train_ds, validation_data=test_ds, epochs=20)

loss, acc = lstm_model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 192ms/step - accuracy: 0.2796 - loss: 1.3818 - val_accuracy: 0.2770 - val_loss: 1.3695
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 178ms/step - accuracy: 0.3970 - loss: 1.3045 - val_accuracy: 0.4414 - val_loss: 1.0961
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 182ms/step - accuracy: 0.4550 - loss: 1.1643 - val_accuracy: 0.5246 - val_loss: 1.0056
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 184ms/step - accuracy: 0.5352 - loss: 1.0091 - val_accuracy: 0.5479 - val_loss: 0.9524
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 182ms/step - accuracy: 0.5809 - loss: 0.9290 - val_accuracy: 0.6039 - val_loss: 0.8673
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 182ms/step - accuracy: 0.6190 - loss: 0.8447 - val_accuracy: 0.5779 - val_loss: 0.8876
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 183ms/step - accuracy: 0.6571 - loss: 0.7373 - val_accuracy: 0.6012 - val_loss: 0.8564
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 182ms/step - accuracy: 0.7142 - loss: 0.6521 - val_accuracy: 0

# 9. Build BiLSTM

In [40]:
bi_lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, mask_zero=True),
    tf.keras.layers.SpatialDropout1D(0.5),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, dropout = 0.3, recurrent_dropout = 0.3 )),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(categories), activation='softmax')
])

bi_lstm_model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

bi_lstm_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_6                  │ ?                           │               0 │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 10. Train and Evaluate BiLSTM model

In [41]:
bi_lstm_model.fit(train_ds, validation_data=test_ds, epochs=20)

loss, acc = bi_lstm_model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 32s 710ms/step - accuracy: 0.2875 - loss: 1.3745 - val_accuracy: 0.3083 - val_loss: 1.3548
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 24s 664ms/step - accuracy: 0.4493 - loss: 1.2722 - val_accuracy: 0.5200 - val_loss: 1.1553
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 24s 659ms/step - accuracy: 0.6243 - loss: 0.9864 - val_accuracy: 0.6478 - val_loss: 0.8839
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 629ms/step - accuracy: 0.7359 - loss: 0.7010 - val_accuracy: 0.7430 - val_loss: 0.6409
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 645ms/step - accuracy: 0.7988 - loss: 0.5645 - val_accuracy: 0.7776 - val_loss: 0.5795
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 641ms/step - accuracy: 0.8697 - loss: 0.3942 - val_accuracy: 0.7803 - val_loss: 0.5733
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 615ms/step - accuracy: 0.9012 - loss: 0.3081 - val_accuracy: 0.8009 - val_loss: 0.5464
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 615ms/step - accuracy: 0.9211 - loss: 0.2386 - val_accu

# 11. Build GRU model

In [44]:
gru_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, mask_zero=True),
    tf.keras.layers.SpatialDropout1D(0.5),
    tf.keras.layers.GRU(64, dropout = 0.3, recurrent_dropout = 0.3 ),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(categories), activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

gru_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d_8                  │ ?                           │               0 │
│ (SpatialDropout1D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_1 (GRU)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_14 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_20 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_15 (Dropout)                 │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_21 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 12. Train and Evalaute Gru model

In [45]:
gru_model.fit(train_ds, validation_data=test_ds, epochs=20)

loss, acc = gru_model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 163ms/step - accuracy: 0.2667 - loss: 1.3808 - val_accuracy: 0.3382 - val_loss: 1.3745
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 147ms/step - accuracy: 0.3584 - loss: 1.3501 - val_accuracy: 0.3595 - val_loss: 1.3421
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 149ms/step - accuracy: 0.4471 - loss: 1.2473 - val_accuracy: 0.4554 - val_loss: 1.2291
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 146ms/step - accuracy: 0.5397 - loss: 1.0570 - val_accuracy: 0.5173 - val_loss: 1.1000
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 148ms/step - accuracy: 0.6278 - loss: 0.9104 - val_accuracy: 0.5632 - val_loss: 1.0021
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 149ms/step - accuracy: 0.6876 - loss: 0.7750 - val_accuracy: 0.5959 - val_loss: 0.9773
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 153ms/step - accuracy: 0.7727 - loss: 0.6218 - val_accuracy: 0.6079 - val_loss: 0.9576
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 147ms/step - accuracy: 0.8077 - loss: 0.5301 - val_accuracy: 0.